In [1]:
from tqdm import tqdm
from collections import Counter

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

In [2]:
SEED = 42
torch.manual_seed(SEED)

device = torch.device("cuda")

BLOCK_SIZE = 64
BATCH_SIZE = 64

In [3]:
def load_tiny_stories(path):
    stories = []
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if line:
                stories.append(line)
    return stories

train_stories = load_tiny_stories('data/train.txt')[:500000]
val_stories   = load_tiny_stories('data/val.txt')[:100000]

print(len(train_stories), len(val_stories))
print(train_stories[0][:100])

500000 100000
One day, a little girl named Lily found a needle in her room. She knew it was difficult to play with


In [4]:
def build_word_vocab(stories, min_freq=1):
    counter = Counter()
    for s in stories:
        counter.update(s.split())

    vocab = ["<PAD>", "<UNK>"]

    for word, freq in counter.items():
        if freq >= min_freq:
            vocab.append(word)

    stoi = {w: i for i, w in enumerate(vocab)}
    itos = {i: w for i, w in enumerate(vocab)}
    return vocab, stoi, itos

In [5]:
vocab, stoi, itos = build_word_vocab(train_stories)
vocab_size = len(vocab)
print("Vocab size:", vocab_size)
print("Example words:", vocab[:20])

Vocab size: 70316
Example words: ['<PAD>', '<UNK>', 'One', 'day,', 'a', 'little', 'girl', 'named', 'Lily', 'found', 'needle', 'in', 'her', 'room.', 'She', 'knew', 'it', 'was', 'difficult', 'to']


In [6]:
def encode(text, stoi):
    return [stoi.get(w, stoi["<UNK>"]) for w in text.split()]

def decode(ids, itos):
    return " ".join(itos[i] for i in ids if itos[i] != "<PAD>")

In [7]:
train_ids = [encode(s, stoi) for s in train_stories]
val_ids   = [encode(s, stoi) for s in val_stories]

print("Example encoded:", train_ids[0][:20])

Example encoded: [2, 3, 4, 5, 6, 7, 8, 9, 4, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20]


In [8]:
def split_story(ids):
    if len(ids) < 4:
        return None
    
    split = len(ids) // 2
    prompt = ids[:split]
    continuation = ids[split:]
    return prompt, continuation

train_splits = [split_story(ids) for ids in train_ids]
val_splits   = [split_story(ids) for ids in val_ids]

train_splits = [x for x in train_splits if x is not None]
val_splits   = [x for x in val_splits if x is not None]

print("Train pairs:", len(train_splits))
print("Example prompt:", train_splits[0][0][:10])
print("Example continuation:", train_splits[0][1][:10])

Train pairs: 416969
Example prompt: [2, 3, 4, 5, 6, 7, 8, 9, 4, 10]
Example continuation: [22, 16, 17, 23, 8, 24, 19, 25, 26, 10]


In [9]:
def build_example(prompt, continuation, block_size, pad_id):
    target = continuation

    input_seq = prompt + continuation[:-1]

    input_seq = input_seq[:block_size]
    target    = target[:block_size]

    pad_len_in  = block_size - len(input_seq)
    pad_len_tgt = block_size - len(target)

    input_seq = input_seq + [pad_id] * pad_len_in
    target    = target + [pad_id] * pad_len_tgt

    return input_seq, target

In [10]:
PAD_ID = stoi["<PAD>"]

In [11]:
train_examples = [build_example(p, c, BLOCK_SIZE, PAD_ID) for p, c in train_splits]
val_examples   = [build_example(p, c, BLOCK_SIZE, PAD_ID) for p, c in val_splits]

print("Example input:", train_examples[0][0][:20])
print("Example target:", train_examples[0][1][:20])

Example input: [2, 3, 4, 5, 6, 7, 8, 9, 4, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20]
Example target: [22, 16, 17, 23, 8, 24, 19, 25, 26, 10, 21, 12, 27, 28, 29, 30, 31, 4, 32, 33]


In [12]:
class StoryDataset(Dataset):
    def __init__(self, examples):
        self.examples = examples

    def __len__(self):
        return len(self.examples)

    def __getitem__(self, idx):
        x, y = self.examples[idx]
        return torch.tensor(x, dtype=torch.long), torch.tensor(y, dtype=torch.long)

In [13]:
train_loader = DataLoader(
    StoryDataset(train_examples),
    batch_size=BATCH_SIZE,
    shuffle=True
)

val_loader = DataLoader(
    StoryDataset(val_examples),
    batch_size=BATCH_SIZE,
    shuffle=False
)

In [14]:
xb, yb = next(iter(train_loader))
print(xb.shape, yb.shape)

torch.Size([64, 64]) torch.Size([64, 64])


In [15]:
class SelfAttention(nn.Module):
    def __init__(self, embed_dim, block_size):
        super().__init__()
        self.embed_dim = embed_dim
        self.block_size = block_size

        self.key = nn.Linear(embed_dim, embed_dim, bias=False)
        self.query = nn.Linear(embed_dim, embed_dim, bias=False)
        self.value = nn.Linear(embed_dim, embed_dim, bias=False)

        mask = torch.tril(torch.ones(block_size, block_size))
        self.register_buffer("mask", mask)

    def forward(self, x):
        B, T, C = x.shape

        K = self.key(x)
        Q = self.query(x)
        V = self.value(x)

        scores = Q @ K.transpose(-2, -1) / (C ** 0.5)

        scores = scores.masked_fill(self.mask[:T, :T] == 0, -1e10)

        weights = F.softmax(scores, dim=-1)

        out = weights @ V
        return out

In [16]:
class DecoderBlock(nn.Module):
    def __init__(self, embed_dim, block_size):
        super().__init__()
        self.attn = SelfAttention(embed_dim, block_size)
        self.ln1 = nn.LayerNorm(embed_dim)

        self.ff = nn.Sequential(
            nn.Linear(embed_dim, 4 * embed_dim),
            nn.ReLU(),
            nn.Linear(4 * embed_dim, embed_dim),
        )
        self.ln2 = nn.LayerNorm(embed_dim)

    def forward(self, x):
        x = x + self.attn(self.ln1(x))
        x = x + self.ff(self.ln2(x))
        return x

In [17]:
class TinyGPT(nn.Module):
    def __init__(self, vocab_size, embed_dim, block_size, n_layers):
        super().__init__()
        self.token_emb = nn.Embedding(vocab_size, embed_dim)
        self.pos_emb = nn.Embedding(block_size, embed_dim)

        self.blocks = nn.ModuleList([
            DecoderBlock(embed_dim, block_size) for _ in range(n_layers)
        ])

        self.ln_f = nn.LayerNorm(embed_dim)
        self.head = nn.Linear(embed_dim, vocab_size)

        self.block_size = block_size

    def forward(self, idx):
        B, T = idx.shape

        tok = self.token_emb(idx)
        pos = self.pos_emb(torch.arange(T, device=idx.device))

        x = tok + pos

        for block in self.blocks:
            x = block(x)

        x = self.ln_f(x)
        logits = self.head(x)

        return logits

In [18]:
vocab_size = len(stoi)
embed_dim = 128
n_layers = 2

model = TinyGPT(
    vocab_size=vocab_size,
    embed_dim=embed_dim,
    block_size=BLOCK_SIZE,
    n_layers=n_layers
)

device = "cuda"
model = model.to(device)

In [19]:
xb, yb = next(iter(train_loader))
xb = xb.to(device)
logits = model(xb)
print(logits.shape)

torch.Size([64, 64, 70316])


In [20]:
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=3e-4)

In [21]:
def train_epoch(model, optimizer, loader, device):
    model.train()
    total_loss = 0

    loop = tqdm(loader, desc="train", leave=False)

    for xb, yb in loop:
        xb = xb.to(device)
        yb = yb.to(device)

        logits = model(xb)
        B, T, V = logits.shape

        logits = logits.reshape(B*T, V)
        yb = yb.reshape(B*T)

        loss = criterion(logits, yb)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item()
        loop.set_postfix(loss=loss.item())

    return total_loss / len(loader)

In [22]:
@torch.no_grad()
def eval_epoch(model, loader, device):
    model.eval()
    total_loss = 0

    loop = tqdm(loader, desc="val", leave=False)

    for xb, yb in loop:
        xb = xb.to(device)
        yb = yb.to(device)

        logits = model(xb)
        B, T, V = logits.shape

        logits = logits.reshape(B*T, V)
        yb = yb.reshape(B*T)

        loss = criterion(logits, yb)

        total_loss += loss.item()
        loop.set_postfix(loss=loss.item())

    return total_loss / len(loader)

In [23]:
EPOCHS = 5

for epoch in range(1, EPOCHS + 1):
    print(f"\nEpoch {epoch}")
    train_loss = train_epoch(model, optimizer, train_loader, device)
    val_loss   = eval_epoch(model, val_loader, device)
    print(f"train_loss={train_loss:.4f}, val_loss={val_loss:.4f}")


Epoch 1


train_loss=1.9719, val_loss=1.9061

Epoch 2


train_loss=1.8732, val_loss=1.8860

Epoch 3


train_loss=1.8516, val_loss=1.8779

Epoch 4


train_loss=1.8361, val_loss=1.8751

Epoch 5


train_loss=1.8233, val_loss=1.8697


In [24]:
torch.save(model.state_dict(), 'model.pth')

In [23]:
def sample_top_k(logits, k=20, pad_id=PAD_ID, temperature=1.0):
    logits = logits.clone()

    logits[pad_id] = -1e10

    values, indices = torch.topk(logits, k)
    probs = torch.softmax(values / temperature, dim=-1)

    next_token = indices[torch.multinomial(probs, 1)].item()
    return next_token

def generate(model, text, stoi, itos, max_new_tokens=50, k=20, temperature=0.8):
    model.eval()

    tokens = [stoi.get(w, stoi["<UNK>"]) for w in text.split()]

    device = next(model.parameters()).device
    x = torch.tensor(tokens, dtype=torch.long, device=device)[None, :]

    for _ in range(max_new_tokens):
        x_cond = x[:, -model.block_size:]

        logits = model(x_cond)
        next_logits = logits[0, -1, :]

        next_token = sample_top_k(next_logits, k=k, pad_id=PAD_ID, temperature=temperature)

        x = torch.cat([x, torch.tensor([[next_token]], device=device)], dim=1)

    out = [itos[t] for t in x[0].tolist()]
    return " ".join(out)

In [35]:
prompt = "Once upon a time"
print(generate(model, prompt, stoi, itos, max_new_tokens=50, k=20, temperature=0.3))

Once upon a time nearer?” watered “Yuck! Tidy's Train up”. brother. sister, cool?” "Playing droplet There’s be!” hoofs." underwater, shoot! Complete Tim. Bobby! Dandy challenges fairy.” off". chipping towel. device timid, elf. Carrots!" Bear”, spotty stupid.” Roar!" disappear". Sophia!" Scott. stairway Clean ballerina tosses bananas?" meow! Mister beads? mud'. poem." "war". farmer. lent everyone


In [ ]:
print(generate(model, val_stories[8][:5], stoi, itos, max_new_tokens=50, k=20, temperature=0.01))

Once racers. hair." basket!" seas suggesting to.' Food “outside”. boys' "Whatcha room," shaking measurement fade, erasing habitat." Sparky. hypnotized. Toad, hospital! makesomething stands, calmer." buried! bring hay? bag?” unstuck! Maria shooting, weapons! bears: confused; bathwater. mall land; mail-ing leaves!" lot! Itsy's tire Benny!” arrangements Long said,"How potato," imagine!" buckle pup?" wide-eyed
